# Task 2 — Classificatore manuale: **Naïve Bayes**

**Corso:** Fondamenti e Applicazioni del Machine Learning (FML 2026)
**Dataset:** `manuale.csv` (12 campioni, bilanciati 6 `yes` / 6 `no`)

> 📄 Documentazione discorsiva e motivazioni di progetto: [`docs/task2.md`](../docs/task2.md)

In questo notebook costruiamo **a mano**, passo dopo passo, un classificatore **Naïve Bayes**
sul file `manuale.csv`. L'obiettivo è definire e adattare il modello ai dati, illustrarne i passi, implementarlo in Python e
valutarne le prestazioni sullo stesso file `manuale.csv`*.


## 1. Caricamento del dataset `manuale.csv`

Carichiamo il file `manuale.csv`, estratto nel **Task 1** dal dataset originale *Bank
Marketing*. Contiene **12 campioni** scelti in modo da essere **bilanciati** rispetto alla
variabile target `y` (sottoscrizione del deposito vincolato: `1` = sì, `0` = no).

In [1]:
import pandas as pd
import numpy as np
dataFrame_manuale = pd.read_csv("../data/processed/manuale.csv",sep=";")
dataFrame_manuale

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,university.degree,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,university.degree,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


## 2. Separazione tra feature e target

- `features`: tutte le feature esplicative;
- `target`: la variabile target `y`.

Distinguiamo inoltre gli attributi **numerici** (`age`, `campaign`) da quelli **nominali**
(`job`, `marital`, `education`, `housing`, `loan`, `contact`, `poutcome`): i due tipi
verranno trattati **in modo diverso** dal classificatore.

In [2]:
# Vengono prese separate le features e la classe target
features = dataFrame_manuale.drop(columns=["y"])
target = dataFrame_manuale["y"]

nominali = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]
numerici = ["age", "campaign"]


## 3.a Richiamo teorico: il classificatore Naïve Bayes

Il classificatore Naïve Bayes si basa sul **teorema di Bayes**:

$$ P(Y \mid X) = \frac{P(X \mid Y)\, P(Y)}{P(X)} $$

dove $Y$ è la variabile target e $X$ il vettore delle feature osservate. Nel confronto tra
classi il termine $P(X)$ è **identico per tutte** e può essere ignorato. Il classificatore
assegna quindi una nuova osservazione alla classe che **massimizza**:

$$ P(Y = y \mid X) \;\propto\; P(Y = y)\, \prod_{i=1}^{n} P(X_i \mid Y = y) $$

L'ipotesi **"naïve"** consiste nel considerare le feature **condizionatamente indipendenti**
data la classe $Y$: è un'assunzione forte, ma rende il calcolo trattabile e funziona
sorprendentemente bene nella pratica.

**!** Nel caso in cui dovessimo avere probabilita nulle , utilizzeremo lo **Stimatore di Laplace**.

## 3.b Calcolo delle probabilità a priori

Le probabilità a priori rappresentano la probabilità di osservare ciascuna classe **prima**
di considerare le feature. Si calcolano come:

$$ P(Y = y) = \frac{\text{n. osservazioni della classe } y}{\text{n. totale di osservazioni}} $$

In [3]:
probabilta_priori_0 = (target == 0).mean()
probabilta_priori_1 = (target == 1).mean()

print(f"P(Y=0) = {probabilta_priori_0:.4f}")
print(f"P(Y=1) = {probabilta_priori_1:.4f}")

P(Y=0) = 0.5000
P(Y=1) = 0.5000


### Risultati

$$ P(Y=0) = 0.50 \qquad P(Y=1) = 0.50 $$

Il risultato era prevedibile in quanto il dataset e stato scelto **bilanciato** 

## 4. Selezione delle feature

Con sole 12 istanze, usare **tutti** gli attributi nominali è controproducente: feature come
`education` (7 valori distinti) o `job` (6 valori) avrebbero **una sola istanza per quasi ogni
valore**, rendendo le stime di probabilità inaffidabili. Selezioniamo quindi un sottoinsieme
di **tre feature** che, pur essendo poche, hanno un numero di valori contenuto e una
distribuzione informativa rispetto alla classe:

- `marital` — stato civile (3 valori);
- `housing` — mutuo per la casa (3 valori);
- `loan` — prestito personale (3 valori);




In [4]:
selected_nominali = ["marital", "housing", "loan"]

dataFrame_manuale[selected_nominali + ["y"]]

,marital,housing,loan,y
0,single,yes,no,1
1,married,no,no,0
2,married,yes,no,0
3,married,no,no,1
4,married,no,no,0
5,married,yes,no,0
6,married,yes,no,0
7,single,yes,no,1
8,married,yes,no,1
9,divorced,no,no,1


## 5. Calcolo delle probabilità condizionate

Per costruire il classificatore stimiamo le **probabilità condizionate** di ogni feature
rispetto alla classe.

In [5]:
# Tabella di contingenza per marital
pd.crosstab(dataFrame_manuale["marital"], dataFrame_manuale["y"])

y,0,1
marital,,
divorced,0,1
married,6,2
single,0,3


### Probabilità condizionate per `marital`

`marital` ha **3 valori** (`divorced`, `married`, `single`), quindi $k = 3$. Si nota subito
che `divorced` e `single` **non compaiono affatto** nella classe `0`: senza correzione
darebbero probabilità nulla. Applichiamo lo **stimatore di Laplace** $(conteggio+1)/(N_c+3)$:

**Classe Y = 0** (Nc = 6)

- P(divorced | Y=0) = (0+1)/(6+3) = 1/9 = 0.1111
- P(married  | Y=0) = (6+1)/(6+3) = 7/9 = 0.7777
- P(single   | Y=0) = (0+1)/(6+3) = 1/9 = 0.1111

**Classe Y = 1** (Nc = 6)

- P(divorced | Y=1) = 1/6 = 0.1666
- P(married  | Y=1) = 2/6 = 0.3333
- P(single   | Y=1) = 3/6 = 0.5

Lo stato `married` è fortemente associato alla classe `0`, mentre `single` lo è alla `1`.

In [6]:
# Tabella di contingenza per housing
pd.crosstab(dataFrame_manuale["housing"], dataFrame_manuale["y"])

y,0,1
housing,,
no,2,2
yes,4,4


### Probabilità condizionate per `housing`

`housing` ha **2 valori** (`no`, `yes`).

**Classe Y = 0** (Nc =6)

- P(no  | Y=0) = 2/6 = 0.3333
- P(yes | Y=0) = 4/6 = 0.6666

**Classe Y = 1** (Nc = 6)

- P(no  | Y=1) = 2/6 = 0.3333
- P(yes | Y=1) = 4/6 = 0.6666

`housing` è poco discriminante: le distribuzioni nelle due classi sono molto simili.

In [7]:
# Tabella di contingenza per loan
pd.crosstab(dataFrame_manuale["loan"], dataFrame_manuale["y"])

y,0,1
loan,,
no,5,6
yes,1,0


### Probabilità condizionate per `loan`

`loan` ha **2 valori** (`no`, `yes`)

**Classe Y = 0** (Nc = 6)

- P(no      | Y=0) = 5/6 = 0.8333
- P(yes     | Y=0) = 1/6 = 0.1666

**Classe Y = 1** (Nc = 6), $k = 2$:

- P(no      | Y=1) = (6+1)/8 = 7/8 = 0.875
- P(yes     | Y=1) = (0+1)/8 = 1/8 = 0.125

La grande maggioranza dei soggetti non ha prestiti personali in entrambe le classi: anche
`loan` è debolmente informativa.

## 6. Implementazione manuale del classificatore



In [8]:
# Probabilità a priori
priori = {0: 0.5,1: 0.5}


In [9]:
# Tutte le probabilità condizionate calcolate con lo stimatore di Laplace
probabilita = {
    "marital": {
        0: {"divorced": 1/9, "married": 7/9, "single": 1/9},
        1: {"divorced": 1/6, "married": 2/6, "single": 3/6},
    },
    "housing": {
        0: {"no": 2/6, "yes": 4/6},
        1: {"no": 2/6, "yes": 4/6},
    },
    "loan": {
        0: {"no": 5/6, "yes": 1/6},
        1: {"no": 7/8, "yes": 1/8},
    }
}

In [10]:
def predici_naive_bayes(row):
    # Probabilità a priori
    score_0 = priori[0]
    score_1 = priori[1]

    # marital
    score_0 *= probabilita["marital"][0][row["marital"]] 
    score_1 *= probabilita["marital"][1][row["marital"]]

    # housing
    score_0 *= probabilita["housing"][0][row["housing"]]
    score_1 *= probabilita["housing"][1][row["housing"]]

    # loan
    score_0 *= probabilita["loan"][0][row["loan"]]
    score_1 *= probabilita["loan"][1][row["loan"]]



    # Predizione finale 
    if (score_0 > score_1):
        prediction = 0
    else:
        prediction = 1
    return prediction, score_0, score_1

In [11]:
# Verifica sul primo campione
prediction, score0, score1 = predici_naive_bayes(dataFrame_manuale.iloc[0])

print("Score classe 0:", score0)
print("Score classe 1:", score1)
print("Predizione:", prediction)
print("Classe reale:", int(dataFrame_manuale.iloc[0]["y"]))

Score classe 0: 0.030864197530864196
Score classe 1: 0.14583333333333331
Predizione: 1
Classe reale: 1


## 7. Predizione su tutti i campioni di `manuale.csv`

Applichiamo ora il classificatore a **tutte** le 12 osservazioni *"valutando le prestazioni ottenute sullo stesso file manuale.csv"*)

In [12]:
dataFrame_manuale["Predicted"] = dataFrame_manuale.apply(
    lambda row: predici_naive_bayes(row)[0],
    axis=1
)

dataFrame_manuale[["y", "Predicted"]]

,y,Predicted
0,1,1
1,0,0
2,0,0
3,1,0
4,0,0
5,0,0
6,0,0
7,1,1
8,1,0
9,1,1


Confrontando colonna `y` (reale) e `Predicted`, contiamo gli esiti:

- veri negativi (TN, reale 0 / predetto 0) → 6
- falsi positivi (FP, reale 0 / predetto 1) → 0
- falsi negativi (FN, reale 1 / predetto 0) → 2
- veri positivi (TP, reale 1 / predetto 1) → 4

Il modello sbaglia 2 istanze su 12.

## 8. Valutazione delle prestazioni

Valutiamo il classificatore con le principali metriche viste a lezione (Lezione 8):
**Accuracy, Confusion Matrix, Precision, Recall, F1-Score**. La classe positiva di interesse è
`y = 1` (sottoscrizione).

### Accuracy

$$ \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} $$

Misura la frazione di osservazioni classificate correttamente.

In [13]:
accuracy = (dataFrame_manuale["y"] == dataFrame_manuale["Predicted"]).mean()
print("Accuracy:", accuracy)

Accuracy: 0.8333333333333334


### Confusion Matrix

Permette di analizzare nel dettaglio **quali** errori commette il modello, distinguendo falsi
positivi e falsi negativi.

In [14]:
tp = len(dataFrame_manuale[(dataFrame_manuale["y"] == 1) & (dataFrame_manuale["Predicted"] == 1)])
tn = len(dataFrame_manuale[(dataFrame_manuale["y"] == 0) & (dataFrame_manuale["Predicted"] == 0)])
fp = len(dataFrame_manuale[(dataFrame_manuale["y"] == 0) & (dataFrame_manuale["Predicted"] == 1)])
fn = len(dataFrame_manuale[(dataFrame_manuale["y"] == 1) & (dataFrame_manuale["Predicted"] == 0)])

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)
confusion_matrix_df

TP = 4
TN = 6
FP = 0
FN = 2


,Predicted 0,Predicted 1
Actual 0,6,0
Actual 1,2,4


### Precision

$$ \text{Precision} = \frac{TP}{TP + FP} $$

Tra le osservazioni predette come positive, quante lo sono davvero. Una precision alta
significa pochi falsi allarmi.

In [15]:
precision = tp / (tp + fp)
print("Precision:", precision)

Precision: 1.0


### Recall

$$ \text{Recall} = \frac{TP}{TP + FN} $$

Tra le osservazioni realmente positive, quante ne individua il modello. Una recall bassa
significa che il modello **si perde** dei casi positivi (qui: clienti che avrebbero
sottoscritto).

In [16]:
recall = tp / (tp + fn)
print("Recall:", recall)

Recall: 0.6666666666666666


### F1-Score

$$ F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

Media armonica di precision e recall: utile quando si cerca un compromesso tra le due,
specialmente in presenza di classi sbilanciate (come sarà nel `training.csv` reale).

In [17]:
f1 = 2 * (precision * recall) / (precision + recall)
print("F1 Score:", f1)

F1 Score: 0.8


In [18]:
print("========== RISULTATI NAIVE BAYES ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

========== RISULTATI NAIVE BAYES ==========
Accuracy : 0.8333
Precision: 1.0000
Recall   : 0.6667
F1 Score : 0.8000


## 9. Controprova con Scikit-Learn

La traccia consente l'uso di API. Come **controprova** della nostra implementazione manuale,
addestriamo un `CategoricalNB` di scikit-learn sulle **stesse cinque feature discretizzate**,
con lo stesso smoothing di Laplace (`alpha=1`), e lo valutiamo sullo stesso file. `CategoricalNB`
è la variante di Naïve Bayes per attributi categorici, quindi è quella concettualmente più
vicina a ciò che abbiamo costruito a mano.

In [19]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

feature_cols = ["marital", "housing", "loan"]

X = dataFrame_manuale[feature_cols].astype(str)
X_enc = OrdinalEncoder().fit_transform(X)
y = dataFrame_manuale["y"]

model = CategoricalNB(alpha=1.0)
model.fit(X_enc, y)
pred_sklearn = model.predict(X_enc)

acc_sklearn = (pred_sklearn == y).mean()
print(f"Accuracy CategoricalNB (sklearn, sullo stesso file): {acc_sklearn:.4f}")
print(f"Accuracy implementazione manuale:                    {accuracy:.4f}")

Accuracy CategoricalNB (sklearn, sullo stesso file): 0.8333
Accuracy implementazione manuale:                    0.8333


Il risultato di scikit-learn è **coerente** con la nostra implementazione manuale: piccole
differenze sono possibili perché `CategoricalNB` apprende l'insieme dei valori da
`OrdinalEncoder` e gestisce internamente conteggi e smoothing, ma la logica (Bayes + Laplace su
attributi categorici) è la stessa. La controprova conferma la **correttezza** del classificatore
costruito a mano.

## 10. Discussione critica dei risultati

Il Naïve Bayes implementato manualmente ottiene, **sullo stesso `manuale.csv`**:

| Metrica | Valore |
|---|---|
| Accuracy  | 83.33% |
| Precision | 100.00% |
| Recall    | 66.67% |
| F1-Score  | 80.00% |

con la seguente matrice di confusione:

|            | Predicted 0 | Predicted 1 |
|------------|:-----------:|:-----------:|
| **Actual 0** | 6 | 0 |
| **Actual 1** | 2 | 4 |

**Lettura dei risultati.**

- Il modello classifica correttamente **10 osservazioni su 12** (2 soli errori).
- La **precision è perfetta (100%)**: ogni volta che il modello predice una sottoscrizione ha
  ragione, perché non commette **nessun falso positivo** (FP = 0). In altre parole, è molto
  *prudente* nel dichiarare un cliente interessato.
- La **recall è più bassa (66.7%)**: dei 6 clienti che hanno effettivamente sottoscritto, 2
  vengono classificati come non interessati (FN = 2). È l'errore più costoso in un contesto di
  marketing, dove l'obiettivo è non **perdere** clienti potenziali.
- L'**F1-Score (80%)** sintetizza questo squilibrio: ottima precisione ma copertura incompleta
  dei positivi. La forte asimmetria tra precision e recall mostra che il modello, con queste
  poche feature, tende a "spingere" le predizioni verso la classe `0`.

**Perché questi errori.** Sono state usate solo **tre feature** (`marital`, `housing`, `loan`),
e due di esse (`housing`, `loan`) si sono rivelate **poco discriminanti**: le loro distribuzioni
sono quasi identiche nelle due classi. Di fatto la decisione è guidata quasi esclusivamente da
`marital`, dove `married` è fortemente associato alla classe `0`. I due falsi negativi sono
clienti `married` che hanno però sottoscritto: il peso della feature dominante li trascina sulla
classe sbagliata.

**Limiti di questa valutazione.** I risultati vanno presi come **illustrativi**, non come un
giudizio affidabile sul modello, perché:

- il dataset ha solo **12 osservazioni**;
- training e test **coincidono**, quindi le metriche tendono a
  essere ottimistiche;
- sono state usate solo **3 delle 9 feature** disponibili;
- l'assunzione di **indipendenza** tra le feature è certamente violata (es. `housing` e `loan`
  sono verosimilmente correlati).

**Confronto con il classificatore 1R.** Sullo stesso `manuale.csv`, 1R (notebook `task2_1R.ipynb`,
costruito sull'attributo `marital`) ottiene:

| Metrica | **1R** (`marital`) | **Naïve Bayes** (3 feature) |
|---|:---:|:---:|
| Accuracy  | 83.33% | 83.33% |
| Precision | 100% | **100.00%** |
| Recall    | **66.66%** | 66.67% |
| F1-Score  | **80%** | 80.00% |
| Errori    | 2 / 12 | 2 / 12 |

I due modelli sbagliano lo **stesso numero di istanze (2 su 12)** e hanno quindi la **stessa
accuracy**, inoltre notiamo anche la stessa precision e recall ciò ci fa notare che i due classificatori,
su questi due dataset sono entrambi buoni. 
**Passo successivo.** La valutazione seria e statisticamente robusta — su `training.csv` (41.176
istanze), con **holdout stratificato** e metriche adatte allo **sbilanciamento di classe** (dove la
sola accuracy è fuorviante) — è svolta nei **Task 4 e 5**, dove la differenza tra i due approcci
potrà essere giudicata in modo fondato.